# Notebook 6 — K-Nearest Neighbors (KNN)

**Sprint 7 — Machine Learning Fundamentals for AI/ML Engineers**

**Datasets:** Following the brief's requirement to understand both Classification and Regression, this notebook demonstrates KNN on both:
- **Classification:** Breast Cancer Wisconsin dataset (malignant vs benign) — same dataset used in Notebook 5.
- **Regression:** Diabetes dataset (disease progression score) — same dataset used in Notebook 4.

(Sprint 6 dataset was unavailable when preparing this notebook, so these consistent scikit-learn datasets are used throughout Notebooks 3–6.)

## Step 1 — Understand the Concept

### What is KNN?
K-Nearest Neighbors is a supervised learning algorithm that makes predictions based on **similarity** to other data points, rather than learning an explicit equation like Linear or Logistic Regression. To predict a new point, KNN looks at the **K closest points** in the training data (its "neighbors") and bases the prediction on them.

This makes KNN a **"lazy learner"** — there is no real "training" phase where parameters are fitted; the algorithm simply memorizes the training data and does all its work at prediction time by searching for nearby points.

### Distance
KNN needs a way to measure "closeness" between two data points in feature space. The choice of distance metric directly determines which points count as "neighbors."

### Euclidean Distance
The most common distance metric — the straight-line ("as the crow flies") distance between two points:

`distance(A, B) = sqrt( (a1-b1)^2 + (a2-b2)^2 + ... + (an-bn)^2 )`

### Neighbors
The K training points with the smallest distance to the new point being predicted.

### Choosing K
- **Small K** (e.g., K=1): the prediction is based on just the single nearest point — very sensitive to noise and outliers (high variance, prone to overfitting).
- **Large K:** the prediction is smoothed over many neighbors — more stable but can blur over real local patterns, and if K is too large the model may underfit (start looking more like "always predict the majority class").
- K is typically chosen via cross-validation, trying several values and picking the one with the best validation performance.

### Classification with KNN
For a new point, look at its K nearest neighbors' class labels and predict the **majority class** ("vote").

### Regression with KNN
For a new point, look at its K nearest neighbors' target values and predict their **average** (or sometimes a distance-weighted average).

### Feature Scaling — why KNN requires it
KNN's predictions depend entirely on distance calculations. If one feature has a much larger numeric range than another (e.g., "annual income in dollars" vs "age in years"), that feature will completely dominate the distance calculation, effectively making the other features irrelevant — even if the other features are actually more predictive. **Scaling all features to a comparable range (e.g., standardization) is essential** before using KNN, so every feature contributes fairly to the notion of "distance."

### Advantages
- Simple and intuitive — easy to explain and implement.
- No assumptions about the underlying data distribution (non-parametric).
- Naturally handles multi-class classification.
- Can capture complex, non-linear decision boundaries.

### Limitations
- Slow at prediction time on large datasets (must compute distance to every training point, unless using specialized data structures).
- Very sensitive to irrelevant features and to feature scale.
- Performance degrades in very high-dimensional spaces ("curse of dimensionality" — in high dimensions, all points start to look roughly equally far apart).
- Requires storing the entire training dataset in memory.

### When to use it
- Dataset is small-to-moderate in size.
- Decision boundary is expected to be non-linear.
- A simple, explainable baseline is wanted quickly.

### When not to use it
- Very large datasets where prediction speed matters.
- Very high-dimensional data without dimensionality reduction.
- Data with many irrelevant/noisy features that haven't been filtered out.

## Step 2 — Demonstrate the Concept

**Simple example:** Guessing a house's price by looking at the 5 most similar houses nearby (similar size, similar location, similar age) and averaging their sale prices.

**Mathematical intuition:** Imagine every training data point plotted in space based on its feature values. To classify a brand-new point, draw an expanding circle around it until it contains exactly K training points, then let those K points "vote."

**Real-world example:** A recommendation system suggesting movies based on what similar users (nearest neighbors in "taste space") enjoyed.

**Business example:** Flagging a new credit card transaction as potentially fraudulent by comparing it to the K most similar historical transactions and checking how many of those were confirmed fraud.

**AI/ML use case (our datasets):**
- Classification: predicting malignant vs benign by comparing a new tumor's measurements to the K most similar tumors already diagnosed.
- Regression: predicting a new patient's disease-progression score by averaging the scores of the K most similar patients (by baseline health measurements).

## Step 3 — Implement the Concept (Python)

### Part A — KNN for Classification (Breast Cancer dataset)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer, load_diabetes
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix,
    mean_squared_error, r2_score
)

np.random.seed(42)

cancer = load_breast_cancer(as_frame=True)
X_clf, y_clf = cancer.data, cancer.target

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)

# Scaling is REQUIRED for KNN - fit on train only
scaler_c = StandardScaler().fit(X_train_c)
X_train_c_scaled = scaler_c.transform(X_train_c)
X_test_c_scaled = scaler_c.transform(X_test_c)

print("Train:", X_train_c.shape, " Test:", X_test_c.shape)

### 3.1 Demonstrating WHY scaling matters for KNN

We compare KNN accuracy with and without scaling.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

# WITHOUT scaling (raw feature values)
knn_unscaled = KNeighborsClassifier(n_neighbors=5)

# Train the model
knn_unscaled.fit(X_train_c, y_train_c)

# Calculate accuracy
acc_unscaled = knn_unscaled.score(X_test_c, y_test_c)

print("Accuracy without scaling:", acc_unscaled)

### 3.2 Choosing K — experimenting with different values

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold

k_values = range(1, 31)
cv_scores = []

# 5-fold stratified cross-validation
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)

    scores = cross_val_score(
        knn,
        X_train_c_scaled,
        y_train_c,
        cv=cv,
        scoring="accuracy"
    )

    cv_scores.append(scores.mean())

# Print results
for k, score in zip(k_values, cv_scores):
    print(f"k={k}: CV Accuracy={score:.4f}")

### 3.3 Final KNN Classifier evaluation on the untouched test set

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

# Create final KNN model using the best k
final_knn_clf = KNeighborsClassifier(n_neighbors=best_k)

# Train the model
final_knn_clf.fit(X_train_c_scaled, y_train_c)

# Make predictions on test data
y_pred_c = final_knn_clf.predict(X_test_c_scaled)

print("Best k:", best_k)
print("Predictions:", y_pred_c)

### 3.4 Manually implementing KNN classification with NumPy (K=5, first test point)

To make sure we understand what `KNeighborsClassifier` is doing internally.

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler

# Scale the classification features
scaler_c = StandardScaler()

X_train_c_scaled = scaler_c.fit_transform(X_train_c)
X_test_c_scaled = scaler_c.transform(X_test_c)

print("Scaled training shape:", X_train_c_scaled.shape)
print("Scaled testing shape:", X_test_c_scaled.shape)

### Part B — KNN for Regression (Diabetes dataset)

In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split

# Load Diabetes dataset
diabetes = load_diabetes(as_frame=True)

# Separate features and target
X_reg = diabetes.data
y_reg = diabetes.target

# Split into training and testing data
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg,
    y_reg,
    test_size=0.2,
    random_state=42
)

print("Training shape:", X_train_r.shape)
print("Testing shape:", X_test_r.shape)

In [ ]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import cross_val_score, KFold
import numpy as np

k_values = range(1, 31)
cv_rmse = []

# 5-fold cross-validation
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

for k in k_values:
    knn_r = KNeighborsRegressor(n_neighbors=k)

    # cross_val_score uses negative RMSE,
    # so convert it back to positive RMSE
    scores = cross_val_score(
        knn_r,
        X_train_r_scaled,
        y_train_r,
        cv=cv,
        scoring="neg_root_mean_squared_error"
    )

    cv_rmse.append(-scores.mean())

# Find the k with the lowest RMSE
best_k_reg = k_values[np.argmin(cv_rmse)]

print("Best k:", best_k_reg)
print("Best CV RMSE:", min(cv_rmse))

In [ ]:
from sklearn.neighbors import KNeighborsRegressor

# Create final KNN regression model using the best k
final_knn_reg = KNeighborsRegressor(n_neighbors=best_k_reg)

# Train the model
final_knn_reg.fit(X_train_r_scaled, y_train_r)

# Make predictions
y_pred_r = final_knn_reg.predict(X_test_r_scaled)

print("Best k:", best_k_reg)
print("Predictions:", y_pred_r)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 6))

plt.scatter(y_test_r, y_pred_r, alpha=0.6)

lims = [
    min(y_test_r.min(), y_pred_r.min()),
    max(y_test_r.max(), y_pred_r.max())
]

# Perfect prediction line
plt.plot(
    lims,
    lims,
    color='red',
    linestyle='--',
    label='Perfect prediction'
)

plt.xlabel("Actual Values")
plt.ylabel("Predicted Values")
plt.title("Actual vs Predicted Values")
plt.legend()
plt.grid(True)

plt.show()

## Step 4 — Interpret the Result

**On feature scaling:** The accuracy comparison in section 3.1 is direct evidence for why KNN "requires" scaling — without it, features measured on a much larger numeric scale (like `mean area`, which can be in the hundreds/thousands) dominate the Euclidean distance calculation, drowning out the contribution of features on a smaller scale (like `mean radius`, roughly 0–2 in the training data shown here). Standardizing every feature to a comparable scale lets each one contribute fairly to the notion of "closeness."

**On choosing K:** The cross-validated accuracy/RMSE curve against K typically shows an initial improvement as K increases from 1 (reducing sensitivity to individual noisy points), then flattens or degrades once K becomes too large (the neighborhood becomes so broad it starts including points that aren't really "similar" anymore, pulling predictions toward the overall average/majority class — a form of underfitting). We selected the K that minimized cross-validated error/maximized cross-validated accuracy, using **only training data** — the test set was not touched during this tuning step, following the Notebook 3 principle.

**On the classification result:** The confusion matrix and precision/recall numbers tell us the same story as in Notebook 5 — in this medical classification context, recall on the malignant class matters more than raw accuracy, because a missed malignant case (false negative) is far more costly than a false alarm.

**On the manual KNN implementation:** Reproducing scikit-learn's prediction by hand (computing Euclidean distances, sorting, taking a majority vote among the K nearest) confirms there is no hidden complexity — `KNeighborsClassifier` really is just doing a distance search and a vote, which is exactly why it is described as a "lazy learner" with no real training phase.

**On the regression result:** R² and RMSE here tell us how well the *average of K nearby patients' outcomes* predicts a new patient's outcome. Compared to the Linear Regression result from Notebook 4 on the same dataset, the two approaches encode very different assumptions — Linear Regression assumes a single global linear relationship holds everywhere, while KNN assumes similar patients (by raw feature distance) will have similar outcomes locally, without assuming any particular global equation. Comparing their test-set R²/RMSE side by side (as we will do more formally in the later Model Comparison notebook) is exactly how you would empirically decide which assumption fits this particular dataset better, rather than assuming one algorithm is always superior.